# 01. 시계열 예측 기초

목표: Nexus를 이해하기 전에 필요한 시계열 기본 개념을 작은 예제로 익힌다.

실행 방법:
1. Jupyter Notebook 또는 VS Code에서 이 파일을 연다.
2. 위에서 아래로 셀을 실행한다.
3. `numpy`, `pandas`, `matplotlib`가 없다면 `pip install numpy pandas matplotlib`을 먼저 실행한다.

이 노트북은 공식 Nexus 구현이 아니다. 숫자 시계열, 이벤트 맥락, MAPE/RMSE를 직관적으로 배우기 위한 축소 실습이다.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 재현 가능한 실험을 위해 난수 시드를 고정한다.
# 같은 코드를 다시 실행해도 같은 노이즈가 만들어져 결과 비교가 쉽다.
rng = np.random.default_rng(7)

## 1. 장난감 시계열 만들기

Nexus가 다루는 현실 데이터는 숫자와 사건 맥락이 함께 있다. 여기서는 주간 재고량처럼 보이는 작은 데이터를 만들고, 특정 주에 이벤트 효과를 넣는다.

In [ ]:
weeks = np.arange(1, 81)

# trend: 시간이 갈수록 값이 조금씩 증가하는 장기 흐름이다.
trend = 100 + weeks * 0.8

# seasonality: 13주 주기로 반복되는 계절성이다.
# sin 함수를 쓰면 반복 패턴을 간단히 만들 수 있다.
seasonality = 8 * np.sin(2 * np.pi * weeks / 13)

# event_effect: 특정 기간에만 값이 위아래로 밀리는 사건 효과다.
event_effect = np.zeros_like(weeks, dtype=float)
event_effect[(weeks >= 28) & (weeks <= 33)] += 12
event_effect[(weeks >= 56) & (weeks <= 61)] -= 10

# noise: 현실 데이터의 작은 흔들림을 흉내 낸다.
noise = rng.normal(0, 2.0, size=len(weeks))

values = trend + seasonality + event_effect + noise

events = []
for week in weeks:
    if 28 <= week <= 33:
        events.append("positive demand shock")
    elif 56 <= week <= 61:
        events.append("policy drag reduces demand")
    else:
        events.append("ordinary market week")

df = pd.DataFrame({"week": weeks, "value": values, "event": events})
df.head()

In [ ]:
ax = df.plot(x="week", y="value", figsize=(10, 4), legend=False, title="Toy time series with event effects")
ax.set_xlabel("week")
ax.set_ylabel("value")
plt.show()

## 2. 예측 구간과 검증 분리

시계열에서는 미래 정보를 섞지 않는 것이 중요하다. 그래서 앞부분은 과거 관측 구간으로, 뒷부분은 검증 구간으로 둔다. 랜덤 셔플을 쓰면 미래가 과거 학습에 섞일 수 있다.

In [ ]:
horizon = 12
train = df.iloc[:-horizon].copy()
valid = df.iloc[-horizon:].copy()

print("train weeks:", int(train.week.min()), "to", int(train.week.max()))
print("valid weeks:", int(valid.week.min()), "to", int(valid.week.max()))

## 3. 기본 예측기와 평가 지표

가장 단순한 기준선은 마지막 값을 그대로 미래에 반복하는 naive forecast다. 또 다른 기준선은 최근 몇 개 값의 평균을 반복하는 moving average forecast다. 단순 기준선은 새 방법이 정말 의미 있는지 판단하는 출발점이다.

In [ ]:
def naive_forecast(history, steps):
    """마지막 관측값을 미래 모든 시점의 예측값으로 사용한다."""
    last_value = float(history[-1])
    return np.repeat(last_value, steps)


def moving_average_forecast(history, steps, window=6):
    """최근 window개 평균을 미래 예측값으로 반복한다."""
    recent_mean = float(np.mean(history[-window:]))
    return np.repeat(recent_mean, steps)


def mape(y_true, y_pred):
    """실제값 대비 절대 오차 비율의 평균이다. 0 나눗셈을 피하기 위해 작은 값을 더한다."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8)))


def rmse(y_true, y_pred):
    """큰 오차에 더 민감한 제곱근 평균 제곱 오차다."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return math.sqrt(np.mean((y_true - y_pred) ** 2))

In [ ]:
history = train["value"].to_numpy()
actual = valid["value"].to_numpy()

pred_naive = naive_forecast(history, horizon)
pred_ma = moving_average_forecast(history, horizon, window=8)

scores = pd.DataFrame([
    {"model": "naive", "MAPE": mape(actual, pred_naive), "RMSE": rmse(actual, pred_naive)},
    {"model": "moving_average", "MAPE": mape(actual, pred_ma), "RMSE": rmse(actual, pred_ma)},
])
scores

## 4. 텍스트 이벤트를 숫자 보정으로 바꾸기

Nexus의 핵심은 텍스트 맥락을 숫자 예측과 분리하지 않는 것이다. 여기서는 LLM 대신 아주 단순한 규칙을 사용해 이벤트 문장을 보정값으로 바꾼다.

In [ ]:
def event_adjustment(event_text):
    """이벤트 문장을 작은 숫자 신호로 바꾼다.

    실제 Nexus라면 이 부분을 LLM 에이전트가 맡는다.
    실습에서는 규칙을 명시해 텍스트 맥락이 예측에 들어가는 위치를 보여준다.
    """
    text = event_text.lower()
    if "positive" in text or "demand shock" in text:
        return 8.0
    if "drag" in text or "reduces" in text:
        return -7.0
    return 0.0


def event_aware_forecast(history, future_events):
    """기본 추세 예측에 이벤트 보정값을 더한다."""
    base = moving_average_forecast(history, len(future_events), window=8)
    adjustments = np.array([event_adjustment(text) for text in future_events])
    return base + adjustments


pred_event = event_aware_forecast(history, valid["event"].tolist())

scores = pd.concat([
    scores,
    pd.DataFrame([{ "model": "event_aware", "MAPE": mape(actual, pred_event), "RMSE": rmse(actual, pred_event)}])
], ignore_index=True)
scores.sort_values("MAPE")

In [ ]:
plot_df = valid[["week", "value"]].copy()
plot_df["naive"] = pred_naive
plot_df["moving_average"] = pred_ma
plot_df["event_aware"] = pred_event

ax = plot_df.plot(x="week", y=["value", "naive", "moving_average", "event_aware"], figsize=(10, 4))
ax.set_title("Validation forecasts")
ax.set_ylabel("value")
plt.show()

## 정리

- 시계열 검증은 시간 순서를 지켜야 한다.
- 단순 기준선은 새 방법의 효과를 판단하는 최소 기준이다.
- 텍스트 이벤트를 숫자 예측에 반영하려면, 이벤트를 구조화된 신호로 바꾸는 단계가 필요하다.
- 다음 노트북에서는 이 아이디어를 Nexus처럼 여러 에이전트 역할로 나누어 구현한다.